In [89]:
from pyspark.sql.functions import *

fact_orders = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/fact_orders"
)

In [90]:
latest_date = fact_orders.select(
    max("order_date")
).collect()[0][0]

print(latest_date)

In [91]:
rfm = (
    fact_orders
    .groupBy("user_id")
    .agg(
        datediff(
            lit(latest_date),
            max("order_date")
        ).alias("recency"),

        count("order_id").alias("frequency"),

        sum("sales_amount").alias("monetary")
    )
)

In [92]:
display(
    rfm.orderBy(desc("monetary"))
)

In [93]:
from pyspark.sql.window import Window

rfm = rfm.withColumn(
    "r_score",
    ntile(4).over(Window.orderBy(col("recency").asc()))
)

rfm = rfm.withColumn(
    "f_score",
    ntile(4).over(Window.orderBy(col("frequency").desc()))
)

rfm = rfm.withColumn(
    "m_score",
    ntile(4).over(Window.orderBy(col("monetary").desc()))
)

In [94]:
rfm = rfm.withColumn(
    "rfm_score",
    concat(
        col("r_score"),
        col("f_score"),
        col("m_score")
    )
)

In [95]:
rfm = rfm.withColumn(
    "segment",
    when(
        (col("r_score") >= 3) &
        (col("f_score") >= 3) &
        (col("m_score") >= 3),
        "Champions"
    )
    .when(
        (col("r_score") >= 3) &
        (col("f_score") >= 2),
        "Loyal Customers"
    )
    .when(
        col("r_score") == 1,
        "At Risk"
    )
    .otherwise("Potential Loyalist")
)

In [96]:
rfm.groupBy("segment").count().show()

In [97]:
rfm.write.mode("overwrite").parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/rfm_customers"
)

In [98]:
rfm_check = spark.read.parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/rfm_customers"
)

print("Rows:", rfm_check.count())

display(rfm_check)